In [1]:
import cv2
import csv
import os
import mediapipe as mp
import numpy as np

In [2]:
cap = cv2.VideoCapture(0)
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=2, min_detection_confidence=0.5)
mp_draw = mp_drawing
mp_hands_conn = mp_hands.HAND_CONNECTIONS
path = "E:\\ML_project\\toa_do_kbb.csv"

In [3]:
recording = False
frame_recorded = 0
lable = ""
lm_list = []
solandaluu = 0

In [4]:
while True:
    success, img = cap.read()
    if not success: break

    imgrgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) 
    result = hands.process(imgrgb)

    key = cv2.waitKey(1) & 0xFF

    if key == 27:
        print("dung camera")
        break

    if ord('a') <= key <= ord('z'):
        pressed = chr(key)
        if pressed != lable:
            solandaluu = 0
        if not recording:
            recording = True
            lable = pressed
            frame_recorded = 0
            lm_list = [lable]
            print(f"dang ghi {lable}")
        
        elif recording and pressed == lable:
            recording = False
            print("da huy")

    if result.multi_hand_landmarks:
        for handlms in result.multi_hand_landmarks:
            mp_draw.draw_landmarks(img, handlms, mp_hands_conn)
            if recording:
                temp_list = []
                for id,lm in enumerate(handlms.landmark):
                    temp_list.append([lm.x,lm.y,lm.z])
                landmarks_arr = np.array(temp_list)

                frame_data = landmarks_arr.flatten().tolist()

                lm_list = [lable] + frame_data

                frame_recorded += 1
                with open(path, mode = 'a', newline='') as f:
                    write = csv.writer(f)
                    write.writerow(lm_list)
                break
    
    else:
        if recording == True:
            recording = False
            frame_recorded = 0
            print("da dung ghi")

    cv2.putText(img, f"Dang ghi: {frame_recorded}", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255) if recording else (0,255,0), 2)
    cv2.imshow("image", img)

e:\ML_project\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


dung camera


In [5]:
cap.release()
cv2.destroyAllWindows()